### Tools

In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
from langchain_groq import ChatGroq

model = ChatGroq(
    model = "qwen/qwen3-32b"
)
model


ChatGroq(metadata={'versions': {'langchain-core': '1.4.6', 'langchain': '1.3.7'}}, output_version=None, profile={'name': 'Qwen3 32B', 'release_date': '2024-12-23', 'last_updated': '2024-12-23', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 40960, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x11279e660>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x11279f0e0>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [3]:
from langchain.chat_models import init_chat_model

agent = init_chat_model(
    model = "groq:qwen/qwen3-32b"
)
response = agent.invoke("Richest man in the world?")
response.content

'<think>\nOkay, so I need to figure out who the richest person in the world is right now. Let me start by recalling what I know. Elon Musk comes to mind because of Tesla and SpaceX, but I think Jeff Bezos was the richest for a while. Maybe it\'s a different person now. I should check recent sources like Forbes or Bloomberg Billionaires Index. \n\nFirst, I remember that the top positions can change frequently due to stock market fluctuations and company valuations. For example, if a company\'s stock price goes up, the CEO\'s wealth might increase. Conversely, if the stock goes down, their net worth decreases. So timing is important here.\n\nLet me think about recent news. I read somewhere that Elon Musk took over as the richest person again, maybe after some of his ventures did well. But then again, when Amazon was doing really well, Jeff Bezos was leading. There\'s also Bernard Arnault from LVMH, who has a huge luxury brand empire. Maybe he\'s up there too. \n\nI should consider the in

### Tools calling
from langchain_tools import tools --> now this tool can be used as a decorator @tool and any funciton can be used a tool

In [4]:
from langchain.tools import tool

@tool
def get_weather(city: str) -> str:
    """Get weather of the city"""
    return f"The weather of {city} is sunny!"


model_with_tools = model.bind_tools([get_weather])


In [6]:
response = model_with_tools.invoke("What is the weather like in Boston?")

response


AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user is asking about the weather in Boston. Let me check the tools available. There\'s a function called get_weather that takes a city parameter. Since the user mentioned Boston, I need to call that function with "Boston" as the city argument. I\'ll make sure the JSON is correctly formatted with the city name in the parameters. Alright, that should get the current weather information for Boston.\n', 'tool_calls': [{'id': '74cnp85hx', 'function': {'arguments': '{"city":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 106, 'prompt_tokens': 153, 'total_tokens': 259, 'completion_time': 0.167465679, 'completion_tokens_details': {'reasoning_tokens': 82}, 'prompt_time': 0.006008611, 'prompt_tokens_details': None, 'queue_time': 0.161257367, 'total_time': 0.17347429}, 'model_name': 'qwen/qwen3-32b', 'system_fingerprint': 'fp_2bfcc54d36', 'service_tier': 'on_

### Tool execution loop

In [10]:
#Step 1: Model Generates tool calls
messages = [{"role": "user", "content" : "What is the weather in Boston?"}]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

#Step 2: Exectue Tools and collect results
for tool_call in ai_msg.tool_calls:
    #Excecute the tool with generated arguments
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

final_response = model_with_tools.invoke(messages)
print(final_response)


content='The weather in Boston today is sunny! 🌞 Let me know if you need more details.' additional_kwargs={'reasoning_content': "Okay, the user asked about the weather in Boston. I called the get_weather function with Boston as the city. The response came back saying it's sunny. Now I need to relay that information to the user in a friendly way. Let me check if there's any additional details I should mention, but the function only provided the weather status. So I'll just state that Boston is sunny today. Make sure the response is clear and matches the user's query.\n"} response_metadata={'token_usage': {'completion_tokens': 118, 'prompt_tokens': 191, 'total_tokens': 309, 'completion_time': 0.221425607, 'completion_tokens_details': {'reasoning_tokens': 93}, 'prompt_time': 0.008001391, 'prompt_tokens_details': None, 'queue_time': 0.159703396, 'total_time': 0.229426998}, 'model_name': 'qwen/qwen3-32b', 'system_fingerprint': 'fp_2bfcc54d36', 'service_tier': 'on_demand', 'finish_reason': '